# CS604 Colab launcher

Runtime: **GPU**. Images stay on Colab disk (`/content/data`). Checkpoints and logs go to the shared Drive folder **CS604_RUNS**.

The GitHub repo is private, so the clone cell will ask for a Personal Access Token (hidden). Do not paste the token into the notebook text or commit it.

**Drive:** everyone must have `CS604_RUNS` under *My Drive* (create it there, or open the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR) → Organize → **Add shortcut to Drive** → My Drive, name it `CS604_RUNS`). A Drive *link* is not a file path Colab can write to.

Change `METHOD`, `TEAM`, and `BRANCH` below before you run.

**Optional pilot sweep (5a):** if `RUN_PILOT = True`, short runs on a 1,000-class subset test each value in the config's `pilot.sweep` block, and the train step uses the winner. The train step also sets `epochs` from the pilot's measured speed so training fits `BUDGET_MIN`. With `RUN_PILOT = False` the launcher behaves as before.

In [1]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"  # must contain scripts/pilot_sweep.py and the method config
METHOD = "full_ft_336"  # linear_probe | full_ft | full_ft_336 | adapter_336 | adaptformer_336 | lora_r8 | bitfit | last_blocks | prototype_probe | prototype_lora_r8
TEAM = "08"
PROTO_PT = f"{DRIVE_OUT}/prototypes.pt"

# Pilot sweep + time budget (optional)
RUN_PILOT = True  # needs a pilot: block in configs/{METHOD}.yaml
PILOT_DIR = ""    # paste an earlier pilot folder on Drive to reuse it instead of rerunning
PILOT_DATA_DIR = "/content/pilot_data"  # local disk; label files only, images are symlinked
BUDGET_MIN = 144  # 2 hr x 48GB, taken as 144 min on an A100 40GB
SAFETY = 0.95     # plan epochs to use at most 95% of the budget

## 1. Clone or pull the repo

In [2]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ··········
Cloning into '/content/repo'...
remote: Enumerating objects: 791, done.
remote: Counting objects: 100% (791/791), done.
remote: Compressing objects: 100% (373/373), done.
remote: Total 791 (delta 426), reused 746 (delta 391), pack-reused 0 (from 0)
Receiving objects: 100% (791/791), 582.51 KiB | 17.13 MiB/s, done.
Resolving deltas: 100% (426/426), done.
/content/repo
ana-branch
2cd125c


## 2. Install requirements

In [3]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

In [4]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

if not Path("/content/drive/MyDrive").is_dir():
    raise FileNotFoundError("Drive did not mount. Rerun this cell and complete the sign-in.")

out = Path(DRIVE_OUT)
if not out.is_dir():
    out.mkdir(parents=True, exist_ok=True)
    print("Created", out)

print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

Mounted at /content/drive
Writing runs to /content/drive/MyDrive/CS604_RUNS
Existing items: ['pilot_full_ft_336_20261006_013706']


## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

[download] snapshot doem1997/cs604-course-data -> /content/data
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0% 0/7 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/2.51k [00:00<?, ?B/s]         

Fetching 7 files:  14% 1/7 [00:00<00:01,  3.81it/s]
Reconstructing (incomplete total...):  59% 2.51k/4.22k [00:00<00:00, 9.60kB/s]
Reconstructing (incomplete total...):  86% 4.22k/4.88k [00:00<00:00, 9.60kB/s]
Reconstructing (incomplete total...):   0% 4.88k/1.05M [00:00<01:49, 9.60kB/s]
Reconstructing (incomplete total...):   0% 1.05M/4.44G [00:00<128:33:47, 9.60kB/s]
Reconstructing (incomplete total...):   0% 1.05M/44.6G [00:00<1290:01:32, 9.60kB/s]
Reconstructing (incomplete total...):   0% 29.4M/44.6G [00:02<50:08, 14.8MB/s]     

Reconstructing (incomplete total...):  12% 5.39G/44.6

## 4b. Build class prototypes (once per backbone)

Van's ~43.9% zero-shot head. Run **once**, then reuse `PROTO_PT` for `prototype_probe` / `prototype_lora_r8`. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/prototype_probe.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## Review settings (table)

Loads `configs/{METHOD}.yaml` (merged with `base.yaml`) so you can check every parameter before spending a GPU hour.

In [ ]:
import pandas as pd

from src.utils import apply_overrides, load_config

methods = ["full_ft", "linear_probe", "lora_r8", "bitfit", "last_blocks", "prototype_probe", "prototype_lora_r8", "adapter", "adaptformer"]
if METHOD not in methods:
    methods.append(METHOD)
merged = {name: load_config(f"configs/{name}.yaml").to_dict() for name in methods}
keys = []
for d in merged.values():
    for k in d:
        if k not in keys:
            keys.append(k)

compare = pd.DataFrame(
    {name: [merged[name].get(k, "") for k in keys] for name in methods},
    index=keys,
)
compare.index.name = "parameter"
print("Resolved parameters per method (base.yaml already merged in):")
display(compare)

chosen = load_config(f"configs/{METHOD}.yaml")
apply_overrides(chosen, data_root=DATA_DIR, output_dir=DRIVE_OUT)
resolved = pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"])
print(f"\nThis run (METHOD={METHOD!r}, plus Colab data_root / output_dir):")
display(resolved)

In [ ]:
import json

import matplotlib.pyplot as plt

TRAIN_CONFIG = f"configs/{METHOD}.yaml"
summary = None
if RUN_PILOT:
    if PILOT_DIR:
        sweep_dir = Path(PILOT_DIR)
        print("Reusing pilot:", sweep_dir)
    else:
        !PYTHONPATH={REPO_DIR} python scripts/pilot_sweep.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} \
            --output_dir {DRIVE_OUT} --pilot_data_dir {PILOT_DATA_DIR}
        pilots = sorted(Path(DRIVE_OUT).glob("pilot_*"), key=lambda p: p.stat().st_mtime)
        if not pilots or not (pilots[-1] / "pilot_summary.json").exists():
            raise RuntimeError("Pilot did not finish. Scroll up to the first error in the script output.")
        sweep_dir = pilots[-1]
        print("Pilot folder:", sweep_dir)
    display(pd.read_csv(sweep_dir / "pilot_sweep.csv"))
    summary = json.loads((sweep_dir / "pilot_summary.json").read_text())
    print("Chosen:", summary["chosen"], "|", summary["reason"])
    TRAIN_CONFIG = str(sweep_dir / "best_config.yaml")

    fig, ax = plt.subplots(figsize=(6, 4))
    for r in summary["results"]:
        log = pd.read_csv(Path(r["run_dir"]) / "epoch_log.csv")
        label = ", ".join(f"{k}={r[k]}" for k in summary["chosen"])
        ax.plot(log["epoch"], log["val_acc"], marker="o", label=label)
    ax.set_xlabel("epoch"); ax.set_ylabel("pilot holdout top-1"); ax.legend(); ax.grid(alpha=0.3)
    plt.show()
else:
    print("No pilot; training with", TRAIN_CONFIG)

In [ ]:
# ===== Head initialisation test (pilot scale, full FT) =====
# Runs 4 short full-FT runs that differ only in the starting head.
# Heads are built from the pilot's FIT images only, so the holdout stays clean.
import gc, importlib.util, json, os
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

import src.train as T
from src.data import build_dataloaders
from src.model import build_model
from src.utils import load_config

HEAD_CONFIG = "configs/full_ft_336.yaml"
BACKBONE_LR = 1e-5          # chosen by the LR pilot
ARMS = [                    # (name, starting head, head_lr)
    ("A_random", "random", 1e-3),
    ("B_prototype", "prototype", 1e-3),
    ("C_probe", "probe", 1e-3),
    ("D_probe_low_head_lr", "probe", 1e-4),
]
NUM_WORKERS = os.cpu_count()  # same for every arm; doesn't change accuracy, only speed
device = torch.device("cuda")

# 1. Same pilot subset and holdout as the LR pilot (seed 0), rebuilt if this is a new session
spec = importlib.util.spec_from_file_location("pilot_sweep", f"{REPO_DIR}/scripts/pilot_sweep.py")
ps = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ps)
pc = dict(load_config(HEAD_CONFIG).pilot)
pilot_root, info = ps.build_pilot_root(
    DATA_DIR, PILOT_DATA_DIR, int(pc["num_classes_subset"]), int(pc["holdout_per_class"]),
    int(pc.get("subset_seed", 0)))
print("Pilot data:", pilot_root, info)

def pilot_cfg(**extra):
    cfg = load_config(HEAD_CONFIG)
    cfg.update({k: v for k, v in pc.items() if k not in ps.PILOT_META_KEYS})  # pilot epochs, warmup
    cfg.update({"data_root": str(pilot_root), "num_workers": NUM_WORKERS, "lr": BACKBONE_LR})
    cfg.update(extra)
    return cfg

# 2. Frozen features (eval transforms) for fit images and holdout images
#    A second data root whose val.txt lists the FIT images lets us reuse the eval loader.
fit_view = Path(PILOT_DATA_DIR) / (Path(pilot_root).name + "_fit_view")
fit_view.mkdir(parents=True, exist_ok=True)
for entry in Path(pilot_root).iterdir():
    if entry.name in ("val.txt", ".done"):
        continue
    link = fit_view / entry.name
    if not link.exists():
        link.symlink_to(entry.resolve(), target_is_directory=entry.is_dir())
(fit_view / "val.txt").write_text((Path(pilot_root) / "train.txt").read_text())

def extract(model, loader, desc):
    feats, labels = [], []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        for images, y, _ in tqdm(loader, desc=desc):
            x = model.forward_features(images.to(device, non_blocking=True))
            feats.append(model.forward_head(x, pre_logits=True).float().cpu())
            labels.append(y.cpu())
    return torch.cat(feats), torch.cat(labels).long()

fe_model = build_model(pilot_cfg()).to(device).eval()
fit_x, fit_y = extract(fe_model, build_dataloaders(pilot_cfg(data_root=str(fit_view), batch_size=128))["val"], "fit features")
hold_x, hold_y = extract(fe_model, build_dataloaders(pilot_cfg(batch_size=128))["val"], "holdout features")
num_classes, dim = fe_model.get_classifier().out_features, fit_x.shape[1]
del fe_model
gc.collect(); torch.cuda.empty_cache()
print(f"fit {tuple(fit_x.shape)}, holdout {tuple(hold_x.shape)}, {fit_y.unique().numel()} classes")

# 3a. Prototype head: unit-length class means, scaled so logits are about 16 * cosine.
#     Classes absent from the pilot get bias -scale (like cosine -1).
def prototype_head(x, y):
    present = y.unique()
    sums = torch.zeros(num_classes, dim).index_add_(0, y, x)
    counts = torch.bincount(y, minlength=num_classes).float()
    scale = 16.0 / x.norm(dim=1).median()
    W = torch.zeros(num_classes, dim)
    W[present] = scale * F.normalize(sums[present] / counts[present, None], dim=1)
    b = torch.full((num_classes,), -16.0)
    b[present] = 0.0
    return W, b

# 3b. Probe head: linear probe on standardised features (your probe's recipe: lr 1e-3,
#     wd 1e-4, label smoothing 0.1, one-cycle), standardisation folded into W and b.
def probe_head(x, y, epochs=30, batch=1024, lr=1e-3, wd=1e-4):
    mu, sd = x.mean(0), x.std(0) + 1e-6
    X, Y = ((x - mu) / sd).to(device), y.to(device)
    head = torch.nn.Linear(dim, num_classes).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    steps = epochs * ((len(X) + batch - 1) // batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.3)
    gen = torch.Generator().manual_seed(0)
    for _ in range(epochs):
        perm = torch.randperm(len(X), generator=gen).to(device)
        for i in range(0, len(X), batch):
            idx = perm[i:i + batch]
            loss = F.cross_entropy(head(X[idx]), Y[idx], label_smoothing=0.1)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    W = head.weight.detach().cpu() / sd
    b = head.bias.detach().cpu() - W @ mu
    return W, b

heads = {"random": None, "prototype": prototype_head(fit_x, fit_y), "probe": probe_head(fit_x, fit_y)}
start_acc = {"random": float("nan")}
for k in ("prototype", "probe"):
    W, b = heads[k]
    start_acc[k] = ((hold_x @ W.T + b).argmax(1) == hold_y).float().mean().item()
print("Holdout accuracy before any training:", {k: round(v, 4) for k, v in start_acc.items()})

# 4. Run the arms. The trainer calls load_prototypes(cfg, model) once after building the
#    model; swap in a version that loads the arm's head, and always restore the original.
orig_load = T.load_prototypes

def head_loader(wb):
    def _load(cfg, model):
        if wb is None:
            return orig_load(cfg, model)
        head = model.get_classifier()
        with torch.no_grad():
            head.weight.copy_(wb[0])
            head.bias.copy_(wb[1])
        print("[head_init] loaded custom starting head")
    return _load

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
headinit_dir = Path(DRIVE_OUT) / f"headinit_full_ft_336_{stamp}"
results = []
try:
    for name, start, head_lr in ARMS:
        print(f"\n===== {name}: start={start}, head_lr={head_lr} =====")
        T.load_prototypes = head_loader(heads[start])
        run_dir = T.train(pilot_cfg(head_lr=head_lr, output_dir=str(headinit_dir / name)))
        r = ps.summarise_run(run_dir, {"arm": name, "head_start": start, "head_lr": head_lr})
        r["start_acc"] = round(start_acc[start], 4)
        results.append(r)
        gc.collect(); torch.cuda.empty_cache()
finally:
    T.load_prototypes = orig_load

res = pd.DataFrame(results)
res.to_csv(headinit_dir / "headinit_results.csv", index=False)
(headinit_dir / "headinit_info.json").write_text(json.dumps(
    {"backbone_lr": BACKBONE_LR, "pilot_data": info, "start_acc": start_acc,
     "num_workers": NUM_WORKERS}, indent=2))
print("Saved to", headinit_dir)
display(res[["arm", "start_acc", "best_val_acc", "best_epoch", "final_val_acc", "final_val_loss", "early_dip", "wall_min"]])

## 5a. Pilot sweep (optional)

Runs only if `RUN_PILOT = True`. Each value in `pilot.sweep` gets a short run (default: 1,000 classes, 3 epochs) scored on a labelled 5-per-class holdout. Writes `pilot_sweep.csv`, `pilot_summary.json` and `best_config.yaml` (the full-run config with the chosen values) to a `pilot_*` folder on Drive. About 6 to 7 min per value at 336 on an A100.

Check the curves before training: an accuracy dip partway through means the lr is too high, and a winner at the edge of the sweep means the sweep should be extended.

In [ ]:
import json

import matplotlib.pyplot as plt

TRAIN_CONFIG = f"configs/{METHOD}.yaml"
summary = None
if RUN_PILOT:
    if PILOT_DIR:
        sweep_dir = Path(PILOT_DIR)
        print("Reusing pilot:", sweep_dir)
    else:
        !python scripts/pilot_sweep.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} \
            --output_dir {DRIVE_OUT} --pilot_data_dir {PILOT_DATA_DIR}
        sweep_dir = sorted(Path(DRIVE_OUT).glob("pilot_*"), key=lambda p: p.stat().st_mtime)[-1]
        print("Pilot folder:", sweep_dir)
    display(pd.read_csv(sweep_dir / "pilot_sweep.csv"))
    summary = json.loads((sweep_dir / "pilot_summary.json").read_text())
    print("Chosen:", summary["chosen"], "|", summary["reason"])
    TRAIN_CONFIG = str(sweep_dir / "best_config.yaml")

    fig, ax = plt.subplots(figsize=(6, 4))
    for r in summary["results"]:
        log = pd.read_csv(Path(r["run_dir"]) / "epoch_log.csv")
        label = ", ".join(f"{k}={r[k]}" for k in summary["chosen"])
        ax.plot(log["epoch"], log["val_acc"], marker="o", label=label)
    ax.set_xlabel("epoch"); ax.set_ylabel("pilot holdout top-1"); ax.legend(); ax.grid(alpha=0.3)
    plt.show()
else:
    print("No pilot; training with", TRAIN_CONFIG)

## 5. Train (checkpoints on Drive)

Uses `TRAIN_CONFIG` from 5a (the pilot's `best_config.yaml`, or `configs/{METHOD}.yaml` without a pilot).

**Time budget:** with a pilot, `epochs` is set from the pilot's measured speed so training fits `SAFETY × BUDGET_MIN`, keeping the cosine schedule complete. Without one, the config's `epochs` is used, so watch the first epoch's time. The run is wrapped in `timeout` as a safety net; `last.pt` is saved every epoch, but an interrupted run should be treated as failed and rerun with fewer epochs.

In [ ]:
import math

train_cfg = load_config(TRAIN_CONFIG)
n_train = sum(1 for _ in open(Path(DATA_DIR) / "train.txt"))
epochs = int(train_cfg.epochs)

if summary is not None:
    epoch_times = []
    for r in summary["results"]:
        log = pd.read_csv(Path(r["run_dir"]) / "epoch_log.csv")
        # skip epoch 1 when possible: it includes first-pass disk reads and warm-up
        times = log["epoch_time_s"].iloc[1:] if len(log) > 1 else log["epoch_time_s"]
        epoch_times.extend(times.tolist())
    img_per_s = summary["pilot_data"]["fit_images"] / float(pd.Series(epoch_times).median())
    min_per_epoch = n_train / img_per_s / 60
    fit_epochs = max(1, math.floor(SAFETY * BUDGET_MIN / min_per_epoch))
    print(f"Pilot speed: {img_per_s:.0f} img/s -> {min_per_epoch:.1f} min per full epoch ({n_train:,} images)")
    print(f"Config epochs: {epochs}; fits in {SAFETY:.0%} of {BUDGET_MIN} min: {fit_epochs}")
    epochs = min(epochs, fit_epochs)
    print(f"Planned: {epochs} epochs, about {epochs * min_per_epoch:.0f} min of training")
else:
    print(f"No pilot timing; using config epochs={epochs}.")

HARD_STOP_MIN = BUDGET_MIN + 5  # small allowance for model loading
!timeout --signal=INT {HARD_STOP_MIN}m python -m src.train --config {TRAIN_CONFIG} \
    --data_root {DATA_DIR} --output_dir {DRIVE_OUT} --epochs {epochs}

In [ ]:
run_dir = sorted(Path(DRIVE_OUT).glob(f"{train_cfg.method}_*"), key=lambda p: p.stat().st_mtime)[-1]
print("Run folder:", run_dir)
log = pd.read_csv(run_dir / "epoch_log.csv")
display(log)

train_min = log["epoch_time_s"].sum() / 60
print(f"Training time: {train_min:.1f} min of {BUDGET_MIN} min budget ({train_min / BUDGET_MIN:.0%})")
if train_min > BUDGET_MIN:
    print("WARNING: over budget. Reduce epochs and rerun.")
if len(log) < epochs or not (run_dir / "run_stats.txt").exists():
    print(f"WARNING: run stopped early ({len(log)}/{epochs} epochs); likely hit the hard stop.")
else:
    print((run_dir / "run_stats.txt").read_text())

## 6. Predict

`CKPT` defaults to the run from step 5 (`best.pt` if it exists, otherwise `last.pt`; the course val split has no labels, so full-data runs only have `last.pt`). To predict from another run, set `CKPT` by hand.

In [ ]:
CKPT = run_dir / ("best.pt" if (run_dir / "best.pt").exists() else "last.pt")
PRED_CONFIG = run_dir / "config.yaml"  # the exact config that was trained
print("Checkpoint:", CKPT)

!python -m src.predict --config {PRED_CONFIG} --ckpt {CKPT} --split val --team {TEAM} --data_root {DATA_DIR} --out_dir {DRIVE_OUT}
!python -m src.predict --config {PRED_CONFIG} --ckpt {CKPT} --split test --team {TEAM} --data_root {DATA_DIR} --out_dir {DRIVE_OUT}